# Fine-Tuning VLM model
- This notebook is dedicated for fine tuning Models

### Installing the required Libraries

In [1]:
!pip install torch torchvision transformers datasets decord av tqdm scikit-learn pandas

### Imports and setup

In [ ]:
import pandas as pd
import sklearn.model_selection import train_test_split
from pathlib import Path
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, Dataset
from transformers import CLIPProcessor, CLIPModel
import decord
from decord import VideoReader, cpu
import numpy as np
from tqdm import tqdm
import os

### Choose and Load Pre-Trained VLM

In [ ]:
model_name = "openai/clip-vit-base-patch32"
processor = CLIPProcessor.from_pretrained(model_name)
base_model = CLIPModel.from_pretrained(model_name)

### Checking for GPU
(CUDA device will be chosen if available or else CPU will be chosen)

In [ ]:
#Checking if GPU is available
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
base_model.to(device)
print(f"Using device: {device}")

### Defining custom dataset

Defining the root path of the data

In [ ]:
#Root directory of the dataset
data_root = Path("/raw_data") 

Function to collect video paths with binary labels

In [ ]:
def collect_videos_with_labels(root_dir):
    video_paths = []
    labels = []

    #Violent videos(Label 1)
    violent_dir = root_dir / "Violent"
    for cam_dir in ["cam1", "cam2"]:
        cam_path = violent_dir/cam_dir
        if cam_path.exists():
            for file in cam_path.glob(".mp4"):
                video_paths.append(str(file))
                labels.append(1)
    
    #Non-Violent videos(Label 0)
    non_violent_dir = root_dir / "NonViolent"
    for cam_dir in ["cam1", "cam2"]:
        cam_path = non_violent_dir/cam_dir
        if cam_path.exists():
            for file in cam_path.glob(".mp4"):
                video_paths.append(str(file))
                labels.append(0)

    return video_paths, labels

Collecting all videos and labels

In [ ]:
all_video_paths, all_labels = collect_videos_with_labels(data_root)

Loading CSVs for additional metadata (action classes)

In [ ]:
violent_csv_path = data_root / "violent-action-classes.csv"
nonviolent_csv_path = data_root / "nonviolent-action-classes.csv"
occurrences_csv_path = data_root / "action-class-occurrences.csv"

Loading if they exist

In [ ]:
action_metadata = {}
if violent_csv_path.exists():
    violent_df = pd.read_csv(violent_csv_path, sep=";")
    for _, row in violent_df.iterrows():
        action_metadata[row["FILE"]] = {"classes": row["ACTION CLASSES"], "violent": 1}

if nonviolent_csv_path.exists():
    nonviolent_df = pd.read_csv(nonviolent_csv_path, sep=";")
    for _, row in nonviolent_df.iterrows():
        action_metadata[row["FILE"]] = {"classes": row["ACTION CLASSES"], "violent": 0}

if occurrences_csv_path.exists():
    occurrences_df = pd.read_csv(occurrences_csv_path, sep=";")
    print("Action Class Occurrences:\n", occurrences_df)

Split into train and validation (80:20)

In [ ]:
train_videos, val_videos, train_labels, val_labels = train_test_split(
    all_video_paths, all_labels, test_size=0.2, stratify=all_labels, random_state=42

print stats

In [ ]:
print(f"Total videos: {len(all_video_paths)} (Violent: {sum(all_labels)}, Non-Violent: {len(all_labels) - sum(all_labels)})")
print(f"Train set: {len(train_videos)} | Val set: {len(val_videos)}")

### Defining Fine-Tuned Model

### Training Setup

### Training Loop

### Model Saving

### Evaluation and Inference